# Загрузка публичных исходных данных

Этот ноутбук **только скачивает** финансовые данные MOEX и дорожные данные METR-LA.
Обработка, расчёт RV, отбор рядов, нормировка и разбиения train/val/test выполняются в отдельном ноутбуке.

Второй набор — **METR-LA**: скорости на дорожных датчиках Лос-Анджелеса.
Это транспортный набор, а не данные электрической сети.


## 1. Настройки

Если MOEX уже скачан нашим загрузчиком, оставьте прежний путь `RAW_ROOT`.
`RECHECK_MOEX_CACHE=True` повторно обходит план и проверяет кэш каждого запроса.
При обычном повторном запуске используется отметка завершённого плана; это не новая проверка всех файлов.
Процессор отдельно проверяет контрольные суммы используемых ответов.


In [1]:
from pathlib import Path
from datetime import date

# Change PROJECT_ROOT only if the kernel starts outside the project.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()

DOWNLOAD_MOEX = True
DOWNLOAD_METR_LA = True
RECHECK_MOEX_CACHE = False

RAW_ROOT = PROJECT_ROOT / "data/raw/moex_iss_2020_2025"
DATE_FROM = date(2020, 1, 1)
DATE_TILL = date(2025, 12, 31)
BOARD = "TQBR"
DAILY_SESSION = 3
REQUEST_DELAY = 0.25
TIMEOUT = 45
MAX_ATTEMPTS = 5

print("Корень проекта:", PROJECT_ROOT)
print("MOEX:", RAW_ROOT)
print("METR-LA:", PROJECT_ROOT / "data/raw/metr_la")


Корень проекта: /home/loktar/Desktop/money_coursework
MOEX: /home/loktar/Desktop/money_coursework/data/raw/moex_iss_2020_2025
METR-LA: /home/loktar/Desktop/money_coursework/data/raw/metr_la


## 2. Финансовый загрузчик

Исторический scope TQBR за 2020–2025 годы: листинг, дневные итоги всех инструментов,
спецификации, границы доступности и все доступные 10-минутные свечи.
Сохраняются исходные JSON-ответы внутри gzip, URL, параметры и контрольные суммы.
Отбора акций и расчёта признаков здесь нет. Текущие спецификации — снимок на дату получения.

Источники: [MOEX ISS](https://iss.moex.com/iss/reference/),
[дневная история](https://iss.moex.com/iss/reference/491),
[свечи](https://iss.moex.com/iss/reference/409).


In [2]:
import calendar
import gzip
import hashlib
import json
import os
import socket
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone

BASE = "https://iss.moex.com/iss"
STATS = {"network_attempts": 0, "cache_hits": 0}
LAST_REQUEST = 0.0


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha(data):
    return hashlib.sha256(data).hexdigest()


def atomic_write(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=path.name + ".", dir=path.parent)
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def save_json(path, value):
    atomic_write(path, json.dumps(value, ensure_ascii=False, indent=2).encode("utf-8"))


def validate_payload(payload, required_blocks):
    if not isinstance(payload, dict) or "error" in payload or "errors" in payload:
        raise ValueError("Expected a successful ISS JSON object")
    for block in required_blocks:
        try:
            records(payload, block)
        except RuntimeError as exc:
            raise ValueError(str(exc)) from exc


def get_json(path, _required_blocks=(), **params):
    global LAST_REQUEST
    url = BASE + "/" + path.lstrip("/") + "?" + urllib.parse.urlencode(sorted(params.items()))
    key = sha(url.encode("utf-8"))
    folder = RAW_ROOT / "requests" / key[:2] / key
    body_path, meta_path = folder / "response.json.gz", folder / "request.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        packed = body_path.read_bytes()
        if meta["url"] != url or sha(packed) != meta["gzip_sha256"]:
            raise RuntimeError("Повреждён кэш: " + str(folder))
        body = gzip.decompress(packed)
        if sha(body) != meta["body_sha256"]:
            raise RuntimeError("SHA-256 исходного ответа не совпадает: " + str(folder))
        payload = json.loads(body)
        validate_payload(payload, _required_blocks)
        STATS["cache_hits"] += 1
        return payload
    for attempt in range(1, MAX_ATTEMPTS + 1):
        pause = REQUEST_DELAY - (time.monotonic() - LAST_REQUEST)
        if pause > 0:
            time.sleep(pause)
        LAST_REQUEST = time.monotonic()
        STATS["network_attempts"] += 1
        try:
            request = urllib.request.Request(url, headers={
                "User-Agent": "MOEX-coursework-raw-download/1.0",
                "Accept": "application/json", "Accept-Encoding": "identity"})
            with urllib.request.urlopen(request, timeout=TIMEOUT) as response:
                body = response.read()
                status, headers = response.status, dict(response.headers.items())
                response_url = response.geturl()
            payload = json.loads(body)
            validate_payload(payload, _required_blocks)
            packed = gzip.compress(body, mtime=0)
            meta = {"url": url, "response_url": response_url, "path": path,
                    "params": params, "received_utc": utc_now(), "http_status": status,
                    "response_headers": headers, "body_bytes": len(body),
                    "body_sha256": sha(body), "gzip_sha256": sha(packed),
                    "raw_storage": "gzip of unchanged HTTP response body bytes"}
            # Metadata is committed last; an interrupted body-only entry is fetched again.
            atomic_write(body_path, packed)
            save_json(meta_path, meta)
            return payload
        except (urllib.error.URLError, TimeoutError, socket.timeout,
                json.JSONDecodeError, ValueError, OSError) as exc:
            status = getattr(exc, "code", None)
            save_json(RAW_ROOT / "last_request_error.json", {
                "utc": utc_now(), "url": url, "attempt": attempt,
                "http_status": status, "error": repr(exc)})
            retryable = status is None or status in {408, 429, 500, 502, 503, 504}
            if not retryable or attempt == MAX_ATTEMPTS:
                raise RuntimeError("Запрос не выполнен; кэш сохранён: " + url) from exc
            time.sleep(min(30, 2 ** (attempt - 1)))


def records(payload, block):
    table = payload.get(block)
    if not isinstance(table, dict) or not isinstance(table.get("columns"), list) or not isinstance(table.get("data"), list):
        raise RuntimeError("В ответе отсутствует ожидаемая таблица: " + block)
    columns = table["columns"]
    if len(set(columns)) != len(columns):
        raise RuntimeError("Повторяющиеся имена колонок: " + block)
    result = []
    for row in table["data"]:
        if not isinstance(row, list) or len(row) != len(columns):
            raise RuntimeError("Некорректная строка: " + block)
        result.append(dict(zip(columns, row)))
    return result


def as_date(value):
    return None if value is None or value == "" else date.fromisoformat(str(value)[:10])


def pages(path, block, **params):
    offset, seen = 0, set()
    for _ in range(100000):
        payload = get_json(path, _required_blocks=(block,), **params, start=offset)
        rows = records(payload, block)
        cursor_rows = records(payload, block + ".cursor") if block + ".cursor" in payload else []
        total = int(cursor_rows[0]["TOTAL"]) if cursor_rows else None
        if not rows:
            if total is not None and offset < total:
                raise RuntimeError("Пустая страница до окончания TOTAL")
            return
        signature = sha(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode("utf-8"))
        if signature in seen:
            raise RuntimeError("Повтор страницы: API мог проигнорировать start")
        seen.add(signature)
        if cursor_rows and int(cursor_rows[0]["INDEX"]) != offset:
            raise RuntimeError("INDEX курсора не совпадает со start")
        yield rows
        offset += len(rows)
        if total is not None and offset >= total:
            return
    raise RuntimeError("Превышен предел страниц")


def dates_between(first, last):
    while first <= last:
        yield first
        first += timedelta(days=1)


def month_ranges(first, last):
    current = first.replace(day=1)
    while current <= last:
        month_end = current.replace(day=calendar.monthrange(current.year, current.month)[1])
        yield max(first, current), min(last, month_end)
        current = month_end + timedelta(days=1)


def checkpoint(state):
    save_json(RUN_DIR / "checkpoint.json", {"updated_utc": utc_now(), **state, "request_stats": STATS})


In [3]:
def download_raw():
    state = {"status": "running", "phase": "listing"}
    started = utc_now()
    checkpoint(state)
    try:
        history_prefix = f"history/engines/stock/markets/shares/boards/{BOARD}"
        candle_prefix = f"engines/stock/markets/shares/boards/{BOARD}/securities"
        intervals, listing_rows = {}, 0
        for batch in pages(history_prefix + "/listing.json", "securities", status="all", limit=1000):
            listing_rows += len(batch)
            for row in batch:
                if row.get("BOARDID") != BOARD:
                    raise RuntimeError("Листинг вернул другой BOARDID")
                first = as_date(row.get("history_from")) or DATE_FROM
                last = as_date(row.get("history_till")) or DATE_TILL
                first, last = max(first, DATE_FROM), min(last, DATE_TILL)
                if first <= last:
                    intervals.setdefault(str(row["SECID"]), []).append((first, last))
        print("Строк исторического листинга:", listing_rows,
              "; SECID с пересечением периода:", len(intervals), flush=True)

        state.update(phase="daily", daily_dates_done=0)
        daily_extent, daily_rows = {}, 0
        total_dates = (DATE_TILL - DATE_FROM).days + 1
        for number, day in enumerate(dates_between(DATE_FROM, DATE_TILL), 1):
            for batch in pages(history_prefix + "/securities.json", "history",
                               date=day.isoformat(), tradingsession=DAILY_SESSION, limit=100):
                for row in batch:
                    if as_date(row.get("TRADEDATE")) != day or row.get("BOARDID") != BOARD:
                        raise RuntimeError("Дневной ответ не соответствует запрошенной дате/режиму")
                    secid = str(row["SECID"])
                    if secid not in daily_extent:
                        daily_extent[secid] = [day, day]
                    daily_extent[secid][1] = day
                daily_rows += len(batch)
            state.update(daily_dates_done=number, last_daily_date=day.isoformat())
            checkpoint(state)
            if number == 1 or number % 25 == 0 or number == total_dates:
                print(f"Дневные данные: {number}/{total_dates}; SECID: {len(daily_extent)}", flush=True)

        # Daily evidence augments listing instead of excluding historical securities.
        for secid, extent in daily_extent.items():
            intervals.setdefault(secid, []).append(tuple(extent))
        universe = sorted(intervals)
        save_json(RUN_DIR / "universe.json", {"created_utc": utc_now(),
            "rule": "TQBR historical listing overlapping requested dates UNION SECID in requested daily history; no liquidity/current-active filter",
            "securities": [{"SECID": secid, "scope_intervals": [[a.isoformat(), b.isoformat()] for a, b in intervals[secid]]} for secid in universe]})
        print("Исторический scope:", len(universe), "SECID. Отбора по ликвидности нет.", flush=True)

        jobs, unavailable = [], []
        state.update(phase="metadata_and_plan", securities_total=len(universe), metadata_done=0)
        for number, secid in enumerate(universe, 1):
            encoded = urllib.parse.quote(secid, safe="")
            specification = get_json(f"securities/{encoded}.json", _required_blocks=("description", "boards"))
            records(specification, "description")
            records(specification, "boards")
            borders_payload = get_json(f"{candle_prefix}/{encoded}/candleborders.json", _required_blocks=("borders",))
            borders = records(borders_payload, "borders")
            available = [row for row in borders if int(row["interval"]) == 10]
            # Union intersecting date intervals before scheduling unique month jobs.
            covered_days = []
            for border in available:
                begin, end = as_date(border.get("begin")), as_date(border.get("end"))
                if begin is None or end is None:
                    raise RuntimeError("Пустые границы interval=10: " + secid)
                for scope_begin, scope_end in intervals[secid]:
                    left, right = max(scope_begin, begin, DATE_FROM), min(scope_end, end, DATE_TILL)
                    if left <= right:
                        covered_days.append((left, right))
            merged = []
            for left, right in sorted(covered_days):
                if merged and left <= merged[-1][1] + timedelta(days=1):
                    merged[-1] = (merged[-1][0], max(merged[-1][1], right))
                else:
                    merged.append((left, right))
            if not merged:
                unavailable.append(secid)
            for left, right in merged:
                jobs.extend((secid, a, b) for a, b in month_ranges(left, right))
            state.update(metadata_done=number, last_metadata_security=secid)
            checkpoint(state)
            print(f"Спецификации и границы: {number}/{len(universe)} — {secid}", flush=True)
        plan = [{"SECID": secid, "from": left.isoformat(), "till": right.isoformat()} for secid, left, right in jobs]
        save_json(RUN_DIR / "candle_plan.json", {"jobs": plan, "no_available_10m_overlap": unavailable})
        print("План 10m:", len(jobs), "отрезков SECID×месяц; без доступного пересечения:", len(unavailable), flush=True)

        candle_rows, empty_jobs = 0, []
        state.update(phase="candles", candle_jobs_total=len(jobs), candle_jobs_done=0)
        for number, (secid, left, right) in enumerate(jobs, 1):
            encoded = urllib.parse.quote(secid, safe="")
            count = 0
            for batch in pages(f"{candle_prefix}/{encoded}/candles.json", "candles",
                               **{"from": left.isoformat() + " 00:00:00",
                                  "till": right.isoformat() + " 23:59:59", "interval": 10}):
                for row in batch:
                    candle_date = as_date(row.get("begin"))
                    if candle_date is None or not left <= candle_date <= right:
                        raise RuntimeError("Свеча вне запрошенного диапазона: " + secid)
                count += len(batch)
            candle_rows += count
            if count == 0:
                empty_jobs.append({"SECID": secid, "from": left.isoformat(), "till": right.isoformat()})
            state.update(candle_jobs_done=number, last_candle_security=secid,
                         last_candle_from=left.isoformat(), candle_rows=candle_rows)
            checkpoint(state)
            print(f"10m: {number}/{len(jobs)} — {secid}, {left}…{right}, строк {count}", flush=True)
        summary = {"status": "completed", "started_utc": started, "finished_utc": utc_now(),
                   "scope": CONFIGURATION, "listing_rows": listing_rows, "securities": len(universe),
                   "daily_rows": daily_rows, "calendar_dates_processed": total_dates,
                   "candle_jobs_completed": len(jobs), "candle_rows": candle_rows,
                   "empty_candle_jobs": empty_jobs, "no_available_10m_overlap": unavailable,
                   "request_stats_this_execution": STATS,
                   "meaning": "All scheduled requests completed, not proof of uninterrupted source candle history."}
        save_json(RUN_DIR / "summary.json", summary)
        checkpoint({**state, "status": "completed"})
        print("План загрузки выполнен. Это не утверждение об отсутствии дыр в источнике.")
        print("Итог:", RUN_DIR / "summary.json")
        return summary
    except BaseException as exc:
        state.update(status="interrupted" if isinstance(exc, KeyboardInterrupt) else "failed", error=repr(exc))
        checkpoint(state)
        print("Остановлено. Повторный запуск использует сохранённый кэш:", RAW_ROOT, flush=True)
        raise


CONFIGURATION = {"schema": "moex-raw-acquisition-v1", "source": BASE, "board": BOARD,
                 "date_from": DATE_FROM.isoformat(), "date_till": DATE_TILL.isoformat(),
                 "daily_tradingsession": DAILY_SESSION, "candle_interval": 10,
                 "candle_session_filter": None, "timezone_of_source_times": "Europe/Moscow",
                 "liquidity_filter": None, "current_active_filter": False}
if DATE_FROM > DATE_TILL:
    raise ValueError("DATE_FROM позже DATE_TILL")
RUN_ID = sha(json.dumps(CONFIGURATION, sort_keys=True).encode("utf-8"))[:16]
RUN_DIR = RAW_ROOT / "runs" / RUN_ID
save_json(RUN_DIR / "configuration.json", CONFIGURATION)


In [4]:
if DOWNLOAD_MOEX:
    completed_summary = RUN_DIR / "summary.json"
    completed_checkpoint = RUN_DIR / "checkpoint.json"
    complete = False
    if completed_summary.is_file() and completed_checkpoint.is_file():
        previous_summary = json.loads(completed_summary.read_text(encoding="utf-8"))
        previous_checkpoint = json.loads(completed_checkpoint.read_text(encoding="utf-8"))
        complete = (previous_summary.get("status") == "completed"
                    and previous_checkpoint.get("status") == "completed"
                    and previous_summary.get("scope") == CONFIGURATION)
    if complete and not RECHECK_MOEX_CACHE:
        summary = previous_summary
        print("План MOEX уже выполнен; сохранённый кэш оставлен без изменений.")
        print("Итог:", completed_summary)
    else:
        summary = download_raw()
else:
    print("Загрузка MOEX выключена настройкой DOWNLOAD_MOEX=False.")


План MOEX уже выполнен; сохранённый кэш оставлен без изменений.
Итог: /home/loktar/Desktop/money_coursework/data/raw/moex_iss_2020_2025/runs/fd83d3d1e091ea15/summary.json


## 3. METR-LA

Исходный HDF5 из папки авторов
[DCRNN](https://github.com/liyaguang/DCRNN) и опубликованные ими файлы дорожного графа.
Проверяются размер, сигнатура HDF5 и закреплённые контрольные суммы.

Соседний `*.source.json` фиксирует источник, время получения и SHA-256.
Контрольные суммы — измеренный снимок; они не объявляются опубликованными авторами датасетов.


In [5]:
def file_sha256(path):
    hasher = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def download_public_file(url, destination, *, expected_sha256=None,
                         expected_bytes=None, kind="binary"):
    """Save a complete official-source file atomically and keep provenance."""
    destination = Path(destination)
    source_path = destination.with_name(destination.name + ".source.json")
    if destination.exists():
        actual_sha = file_sha256(destination)
        if source_path.is_file():
            meta = json.loads(source_path.read_text(encoding="utf-8"))
            if meta.get("url") != url or meta.get("sha256") != actual_sha:
                raise RuntimeError("Existing raw file/source checksum mismatch: " + str(destination))
        elif expected_sha256 and actual_sha == expected_sha256:
            meta = {"url": url, "sha256": actual_sha,
                    "bytes": destination.stat().st_size, "verified_utc": utc_now(),
                    "raw_storage": "unchanged existing file verified against pinned checksum",
                    "checksum_origin": "locally measured snapshot; not an author-published checksum"}
        else:
            raise RuntimeError("Existing raw file has no provenance metadata or trusted pin: " + str(destination))
        if expected_sha256 and actual_sha != expected_sha256:
            raise RuntimeError("Existing raw file does not match the pinned checksum")
        if expected_bytes and destination.stat().st_size != expected_bytes:
            raise RuntimeError("Existing raw file does not match the expected size")
        if kind == "hdf5":
            with destination.open("rb") as handle:
                if handle.read(8) != b"\x89HDF\r\n\x1a\n":
                    raise RuntimeError("Expected an HDF5 file")
        if not source_path.exists():
            save_json(source_path, meta)
        print("Проверенный файл уже скачан:", destination, flush=True)
        return meta

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    try:
        request = urllib.request.Request(url, headers={
            "User-Agent": "Coursework-public-data-download/1.0", "Accept-Encoding": "identity"})
        with urllib.request.urlopen(request, timeout=90) as response, temporary.open("wb") as handle:
            status = getattr(response, "status", None)
            length_header = response.headers.get("Content-Length")
            content_length = int(length_header) if length_header else None
            response_url = response.geturl()
            hasher, size, next_report = hashlib.sha256(), 0, 16 * 1024 * 1024
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                handle.write(chunk)
                hasher.update(chunk)
                size += len(chunk)
                if size >= next_report:
                    print(f"Загрузка {destination.name}: {size / 1024**2:.0f} MiB", flush=True)
                    next_report += 16 * 1024 * 1024
            handle.flush()
            os.fsync(handle.fileno())
        checksum = hasher.hexdigest()
        if content_length is not None and size != content_length:
            raise RuntimeError("Downloaded size differs from Content-Length")
        if expected_bytes is not None and size != expected_bytes:
            raise RuntimeError("Downloaded size differs from the expected size")
        if expected_sha256 is not None and checksum != expected_sha256:
            raise RuntimeError("Downloaded checksum differs from the pinned checksum")
        extra = {}
        if kind == "hdf5":
            with temporary.open("rb") as handle:
                if handle.read(8) != b"\x89HDF\r\n\x1a\n":
                    raise RuntimeError("Expected HDF5; the response may be an HTML error page")
        meta = {"url": url, "response_url": response_url, "received_utc": utc_now(),
                "http_status": status, "bytes": size, "sha256": checksum,
                "checksum_origin": "locally measured snapshot; not an author-published checksum",
                "raw_storage": "unchanged response bytes", **extra}
        os.replace(temporary, destination)
        save_json(source_path, meta)
        print("Скачано:", destination, flush=True)
        return meta
    finally:
        if temporary.exists():
            temporary.unlink()


def download_metr_la():
    root = PROJECT_ROOT / "data/raw/metr_la"
    results = []
    results.append(download_public_file(
        "https://drive.usercontent.google.com/download?id=1pAGRfzMx6K9WWsfDcD1NMbIif0T0saFC&export=download&confirm=t",
        root / "metr-la.h5", kind="hdf5", expected_bytes=57038056,
        expected_sha256="64784b76d6fb8ec9bff4b6decafb354da2bb37840468fdccee5044e511277c05"))
    for name, (size, checksum) in {
        "adj_mx.pkl": (680459, "a35687c6e15aa228dc45027b0ed2a0ea0f4ec78f573deb992c595092d12f61b3"),
        "graph_sensor_ids.txt": (1448, "3ba026caa2e6263ab0ea54b0fa1b125dbfa7216544cd05313b555e826292b990"),
        "graph_sensor_locations.csv": (6341, "eb8ea96e07358b45d0e4ba3b89c2673fa20c54af50150249e627389e749ade6f"),
    }.items():
        results.append(download_public_file(
            "https://raw.githubusercontent.com/liyaguang/DCRNN/master/data/sensor_graph/" + name,
            root / name, expected_sha256=checksum, expected_bytes=size))
    return results


In [6]:
if DOWNLOAD_METR_LA:
    metr_la_download = download_metr_la()
else:
    print("Загрузка METR-LA выключена настройкой DOWNLOAD_METR_LA=False.")


Проверенный файл уже скачан: /home/loktar/Desktop/money_coursework/data/raw/metr_la/metr-la.h5
Проверенный файл уже скачан: /home/loktar/Desktop/money_coursework/data/raw/metr_la/adj_mx.pkl
Проверенный файл уже скачан: /home/loktar/Desktop/money_coursework/data/raw/metr_la/graph_sensor_ids.txt
Проверенный файл уже скачан: /home/loktar/Desktop/money_coursework/data/raw/metr_la/graph_sensor_locations.csv


## 4. Результат

Исходные файлы находятся в `data/raw/`:

| Набор | Путь |
|---|---|
| MOEX | `data/raw/moex_iss_2020_2025/requests/…` и `runs/…` |
| METR-LA | `data/raw/metr_la/metr-la.h5` и файлы графа |

Статус `completed` у MOEX означает завершение плана запросов, а не отсутствие дыр в источнике.
Следующий ноутбук читает эти исходные файлы и создаёт отдельные папки train, val и test.
